# FXS multi-omics GRN pipeline (v5) — Google Colab

Runs the full pipeline from `github.com/adj-04/fxs-research` with **no manual downloads**:
every input is fetched from GEO / STRING / TRRUST, and all results, checkpoints and logs are
written to Google Drive so a disconnect never loses finished work.

**Runtime:** keep the default **Python 3** runtime (CPU is enough). The notebook installs
R 4.6 + Bioconductor 3.23 as binaries via r2u and calls each pipeline part with `Rscript`
— the same fresh-session-per-part pattern as running locally.

| Stage | Script | Typical time (free tier, 2 vCPU) |
|---|---|---|
| Setup | `colab/setup_r.sh`, `colab/install_packages.R` | 10–20 min first time, ~5 min after |
| Fetch | `scripts/01_fetch_data.R` | ~10 min once (cached on Drive) |
| Part 1 | DE, methylation, GENIE3, PANDA | 1–2 h |
| Part 2 | LIONESS, ALPACA, enrichment, validation | 1–2 h |
| Part 3 | GENIE3/ALPACA nulls, centrality, ChEA3/TRRUST | several h — **resumable** |
| 17B | PANDA null | 1–2 h — **resumable** |
| Part 4 | tissue generality, CollecTRI, summary | ~10 min |

Long stages can run in the background (`background=True`); if Colab disconnects, rerun the
setup cells and the same stage — the null loops pick up where they stopped.

## 1. Settings and Google Drive

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = "https://github.com/adj-04/fxs-research.git"
BRANCH   = "main"                                    # or the branch with the v5 scripts
REPO     = "/content/fxs-research"
ROOT     = "/content/drive/MyDrive/fxs-research-run" # data/results/checkpoints live here

os.makedirs(ROOT, exist_ok=True)
os.environ.update({
    "FXS_REPO": REPO,
    "FXS_ROOT": ROOT,
    "R_LIBS_USER": f"{ROOT}/Rlib-4.6",   # source-built packages persist on Drive
    "FXS_CORES": "2",                    # raise on Colab Pro high-CPU runtimes
    # ---- analysis switches (defaults = v5); set to reproduce v4 ----
    "FXS_TARGET_SELECTION": "variance",  # "pvalue" = v4 target choice
    "FXS_N_GENIE3_TARGETS": "5000",
    "FXS_ADJUST_TIMEPOINT": "TRUE",
    "FXS_METH_ADJUST_CELLTYPE": "TRUE",
    "FXS_N_PERM_GENIE3": "20",
    "FXS_N_PERM_PANDA": "20",
    "FXS_N_PERM_ALPACA": "50",
})
print("ROOT:", ROOT)

## 2. Check the VM (disk, memory, CPU) before anything heavy

In [ ]:
!df -h /content /content/drive | sed 's/  */ /g'
!free -h
!nproc
!ps aux --sort=-%mem | head -6

## 3. Get the code

In [ ]:
import subprocess
if os.path.isdir(f"{REPO}/.git"):
    !git -C {REPO} fetch -q origin {BRANCH} && git -C {REPO} checkout -q {BRANCH} && git -C {REPO} pull -q
else:
    !git clone -q -b {BRANCH} {REPO_URL} {REPO}
!git -C {REPO} log --oneline -1

## 4. Install R 4.6 + packages (rerun after every new VM)

In [ ]:
!bash {REPO}/colab/setup_r.sh

In [ ]:
!Rscript {REPO}/colab/install_packages.R 2>&1 | tail -60

## 5. Runner

`run("part1_steps1-8.R")` streams output here and tees it to `logs/` on Drive.
`run(..., background=True)` detaches the job (survives a closed browser tab while the VM lives);
watch it with `tail_log(...)`.

In [ ]:
import subprocess, time, shlex
from pathlib import Path
LOGS = Path(ROOT, "logs"); LOGS.mkdir(exist_ok=True)

def run(script, background=False):
    path = f"{REPO}/scripts/{script}"
    log  = LOGS / (Path(script).stem + ".log")
    cmd  = f"cd {shlex.quote(ROOT)} && Rscript {shlex.quote(path)}"
    if background:
        subprocess.Popen(f"nohup bash -c {shlex.quote(cmd)} > {shlex.quote(str(log))} 2>&1 &", shell=True)
        print(f"Started in background -> {log}\nUse tail_log('{script}')")
        return
    t0 = time.time()
    with open(log, "w") as fh:
        p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            print(line, end=""); fh.write(line); fh.flush()
        p.wait()
    print(f"\n[{script}] exit {p.returncode} in {(time.time()-t0)/60:.1f} min — log: {log}")
    if p.returncode != 0:
        raise RuntimeError(f"{script} failed; see the log above")

def tail_log(script, n=40):
    log = LOGS / (Path(script).stem + ".log")
    running = subprocess.run(f"pgrep -f {shlex.quote(script)}", shell=True,
                             capture_output=True).returncode == 0
    print("RUNNING" if running else "NOT RUNNING", "|", log)
    print(subprocess.run(f"tail -n {n} {shlex.quote(str(log))}", shell=True,
                         capture_output=True, text=True).stdout)

## 6. Fetch all inputs (once — cached on Drive)

In [ ]:
run("01_fetch_data.R")

## 7. Part 1 — DE, methylation, GENIE3, PANDA
Stops early if the FMR1 gate fails. Check the log for the **GENIE3 orientation check** and the
cell-type sensitivity numbers.

In [ ]:
run("part1_steps1-8.R")

## 8. Part 2 — networks, LIONESS (+ limma edge test), ALPACA, enrichment, validation cohort

In [ ]:
!free -h
run("part2_steps9-16.R")

## 9. Part 3 — permutation nulls, centrality, ChEA3/TRRUST
The longest stage. Background mode recommended; if the VM dies, rerun cells 1, 3, 4, 5 and this cell —
GENIE3/ALPACA nulls resume from the last saved permutation.

In [ ]:
run("part3_steps17-19.R", background=True)

In [ ]:
tail_log("part3_steps17-19.R")

## 10. Step 17B — PANDA randomized-prior null (resumable)

In [ ]:
run("17b-run.R", background=True)

In [ ]:
tail_log("17b-run.R")

## 11. Part 4 — tissue generality, CollecTRI TF activity, headline numbers

In [ ]:
run("part4_steps20-22.R")
import pandas as pd
pd.read_csv(f"{ROOT}/results/v5_headline_numbers.csv")

## 12. Bundle results (figures, tables, networks, logs) for download

In [ ]:
!cd {ROOT} && zip -qr /content/fxs_v5_results.zip results figures networks logs data/MANIFEST.csv
!cp /content/fxs_v5_results.zip {ROOT}/
from google.colab import files
files.download("/content/fxs_v5_results.zip")

## 13. Cytoscape (on your laptop)
Colab has no display, so the Cytoscape step runs locally:
1. Unzip `fxs_v5_results.zip` (from cell 12) on your laptop.
2. Open Cytoscape (cap memory in `Cytoscape.vmoptions`, e.g. `-Xmx3g`).
3. In RStudio, run:
```r
setwd("path/to/unzipped/fxs_v5_results")
source("https://raw.githubusercontent.com/adj-04/fxs-research/main/scripts/14_cytoscape_local.R")
```
This applies the FXS style and exports the figure (PNG/PDF), per-module figures, XGMML, the style `.xml` and a `.cys` session.